# AI-assisted workflow

This notebook repeats the workflow of this week’s lesson, but this time an
AI assistant helps us write the code. The division of labour is simple:

- **You decide the GIS part**: which data, which coordinate reference system,
  which spatial predicate, which classes. You are the GIS expert, and these
  decisions go *into* your prompt.
- **The AI assistant helps with the Python part**: boilerplate, syntax you
  don’t remember, restructuring code, explaining error messages.

The responses shown below are *examples*. If you run the same prompts yourself,
you will get different code, and that is fine. What matters is that you can
read the code, run it, and check the result.

This week, we use an AI assistant to speed up and extend the reclassification
and overlay steps of the lesson, using the travel time grid to the Helsinki
railway station.

> **Habit of the week: check against a known answer**
>
> The quickest way to trust new code is to run it on a case where you
> *already know* the result: a number from the lesson, a total that must not
> change, or the output of a simpler (slower) approach. If both agree, you can
> use the new code with confidence.

## Your decisions

- Variable: travel time by public transport during rush hour (`pt_r_tt`, in minutes). Cells with no data (`-1`) are excluded.
- ‘Suitable areas’ (as in the lesson): less than 20 minutes by public transport, *and* more than 4 km walking distance (`walk_d`).
- Travel time classes: 0–15, 15–30, 30–45, 45–60 and more than 60 minutes.
- Study area: the municipality of Helsinki.

In [ ]:
import pathlib
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"

In [ ]:
import geopandas

accessibility_grid = geopandas.read_file(
    DATA_DIRECTORY
    / "helsinki_region_travel_times_to_railway_station"
    / "helsinki_region_travel_times_to_railway_station.gpkg"
)
accessibility_grid = accessibility_grid.loc[accessibility_grid["pt_r_tt"] >= 0]

## Step 1: a faster version of the lesson’s code

In the lesson, we found suitable areas using `apply()` and a lambda function,
which calls a Python function once for each of the more than 13,000 rows.
Let’s ask for a faster version:

**Prompt:**

```text
Rewrite this pandas code without apply(), in a vectorised way:

accessibility_grid["suitable_area"] = accessibility_grid.apply(
    lambda row: int(row["pt_r_tt"] < 20 and row["walk_d"] > 4000), axis=1
)
```

**Example response** (yours may differ):

In [ ]:
accessibility_grid["suitable_area"] = (
    (accessibility_grid["pt_r_tt"] < 20)
    & (accessibility_grid["walk_d"] > 4000)
).astype(int)

**Before you run it**: each comparison creates a column of `True`/`False`
values for all rows at once. `&` combines them element-wise (the vectorised
version of `and`; note the parentheses around each comparison, they are
required). `.astype(int)` turns `True`/`False` into `1`/`0`.

## Check the result

The lesson’s `apply()` version is our known answer. Both versions must produce
exactly the same column, and the lesson found 9 suitable cells:

In [ ]:
import time

start = time.perf_counter()
suitable_area_apply = accessibility_grid.apply(
    lambda row: int(row["pt_r_tt"] < 20 and row["walk_d"] > 4000), axis=1
)
time_apply = time.perf_counter() - start

start = time.perf_counter()
suitable_area_vectorised = (
    (accessibility_grid["pt_r_tt"] < 20)
    & (accessibility_grid["walk_d"] > 4000)
).astype(int)
time_vectorised = time.perf_counter() - start

assert suitable_area_apply.equals(suitable_area_vectorised)
assert accessibility_grid["suitable_area"].sum() == 9

print(f"Identical results. apply(): {time_apply:.3f} s, vectorised: {time_vectorised:.4f} s")

## Step 2: reclassify into your own classes

**Prompt:**

```text
Using mapclassify, classify the column "pt_r_tt" of a GeoDataFrame
`accessibility_grid` into classes with upper bounds of 15, 30, 45 and 60
minutes, plus a last class for everything above 60 minutes. Add a column
"pt_class" with readable labels such as "0-15 min", and show the number of
cells per class.
```

**Example response** (yours may differ):

In [ ]:
import mapclassify

classifier = mapclassify.UserDefined(
    accessibility_grid["pt_r_tt"],
    bins=[15, 30, 45, 60]
)

class_labels = ["0-15 min", "15-30 min", "30-45 min", "45-60 min", "> 60 min"]
accessibility_grid["pt_class"] = [class_labels[class_number] for class_number in classifier.yb]

accessibility_grid["pt_class"].value_counts().sort_index()

**Before you run it**: `mapclassify.UserDefined` takes our class boundaries
as `bins`. Because the largest value is higher than 60, it automatically adds
one more class up to the maximum value. `classifier.yb` holds the class number
(0 to 4) for each row, which we use to pick the label.

One detail worth knowing: are the class boundaries *inclusive*? Is a cell with
exactly 15 minutes in the first or in the second class? Let’s compare with
`pandas.cut()`, where we set this explicitly (`right=True` means the upper
boundary belongs to the class, `15` is in `0-15 min`):

In [ ]:
import numpy
import pandas

pt_class_pandas = pandas.cut(
    accessibility_grid["pt_r_tt"],
    bins=[-numpy.inf, 15, 30, 45, 60, numpy.inf],
    labels=class_labels,
    right=True
)

assert (pt_class_pandas.astype(str) == accessibility_grid["pt_class"]).all()
print("Both classifications agree: upper class boundaries are inclusive.")

## Step 3: overlay and summarise

**Prompt:**

```text
In geopandas, I have a polygon grid `accessibility_grid` with the columns
"pt_class" and "walk_d" (metres), and a polygon `helsinki` (municipality
border), both in EPSG:3067. Keep only the parts of the grid inside Helsinki
(overlay, intersection). Then make a table with one row per "pt_class", with
the number of grid cells and the mean walking distance in km.
```

**Example response** (yours may differ):

In [ ]:
helsinki = geopandas.read_file(
    DATA_DIRECTORY / "helsinki_municipality" / "helsinki_municipality.gpkg"
)
assert accessibility_grid.crs == helsinki.crs, "CRS of the two layers differ"

grid_in_helsinki = accessibility_grid.overlay(helsinki, how="intersection")

summary = (
    grid_in_helsinki
    .groupby("pt_class")
    .agg(
        cells=("pt_class", "size"),
        mean_walk_distance_km=("walk_d", lambda distances: distances.mean() / 1000),
    )
)
summary

**Before you run it**: this combines two things from the lesson, the
overlay (`how="intersection"`) and grouping. `agg()` with *named
aggregations* (`new_column=(input_column, function)`) creates tidy column
names directly.

## Check the result

The number of cells in the summary must add up to the number of cells inside
Helsinki, and no class can have more cells inside Helsinki than in the whole
region:

In [ ]:
assert summary["cells"].sum() == len(grid_in_helsinki)

cells_in_region = accessibility_grid["pt_class"].value_counts()
assert (summary["cells"] <= cells_in_region[summary.index]).all()

print(f"{len(grid_in_helsinki)} of {len(accessibility_grid)} grid cells intersect Helsinki.")

In [ ]:
grid_in_helsinki.plot(column="pt_class", cmap="RdYlBu", linewidth=0, legend=True)

## AI-use log

Whenever you use an AI assistant in this course (in exercises, only where a
task is marked **AI-LLM OK**, see the [course guidelines on AI
tools](https://autogis-site.readthedocs.io/en/latest/course-info/ai-tools.html)),
keep a short log of what you did. For this notebook, it could look like this:

| Step | What I asked | What I changed | How I checked it |
|------|--------------|----------------|------------------|
| 1 | A vectorised version of the lesson’s `apply()` code | Nothing | Compared with the `apply()` result, 9 suitable cells as in the lesson |
| 2 | Classification into my own travel time classes with mapclassify | Nothing | Compared with `pandas.cut()`, checked which class boundaries are inclusive |
| 3 | Overlay with Helsinki and a summary table per class | Nothing | Cell counts add up |